# Convert A Shapefile To GeoParquet

This notebook converts the supraglacial-lakes shapefile archive into GeoParquet. GeoParquet keeps geometry and CRS metadata with the table and is much easier to read from object storage than a multi-file shapefile.

In [1]:
from pathlib import Path


def find_repo_root(start: Path = Path.cwd()) -> Path:
    """Find the repository root when the kernel starts in a subfolder."""
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / ".git").exists() or (path / "downloaded_data").exists():
            return path
    return start


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "downloaded_data"
DATA_DIR.mkdir(exist_ok=True)

import zipfile
import geopandas as gpd

ZIP_PATH = DATA_DIR / "WAIS_Max_Extent.zip"
TARGET_FILE = "WAIS_Max_Extent/WAIS_Jan_2017_Polygons.shp"
PARQUET_FILE = DATA_DIR / "WAIS_Jan_2017_Polygons.parquet"


## Read The Shapefile Directly From The ZIP Archive


In [2]:
uri = f"zip://{ZIP_PATH}!{TARGET_FILE}"
gdf = gpd.read_file(uri)

gdf.head()


,POLY_AREA,CENTROID_X,CENTROID_Y,Area,Shape_Leng,Shape_Area,Bedrock,Coast,GL_ON_IS,Location,...,PolsbyPopp,Fractal,Reock,Schwartzbe,LW_Ratio,Feature_Cl,Elevation,Slope,Speed,geometry
0,2250.000000,-1.122141e+06,-1.179198e+06,2298.467416,240.00000,2250.000000,439.236720,16041.380634,0.000000,Grounded Ice,...,0.490874,1.408355,0.506300,0.700624,0.857143,Lake,360,1.884210,61.69920,"POLYGON ((-1122127.5 -1179172.5, -1122127.5 -1..."
1,25200.000000,-1.122477e+06,-1.178988e+06,25742.726268,2160.00000,25200.000000,415.759238,16085.345499,0.000000,Grounded Ice,...,0.067874,1.319976,0.052796,0.260526,0.114516,Channel,360,1.884210,60.73470,"MULTIPOLYGON (((-1122607.5 -1178917.5, -112265..."
2,675.000000,-1.122930e+06,-1.178760e+06,689.534115,120.00000,675.000000,740.341624,16507.511953,0.000000,Grounded Ice,...,0.589049,1.360778,0.381972,0.767495,0.333333,Lake,388,2.731610,39.97260,"POLYGON ((-1122907.5 -1178767.5, -1122952.5 -1..."
3,11250.000000,-1.123134e+06,-1.178679e+06,11492.233178,930.00000,11250.000000,770.661893,16537.853558,0.000000,Grounded Ice,...,0.163454,1.364722,0.112676,0.404295,0.186667,Lake,388,2.731610,46.55940,"POLYGON ((-1123267.5 -1178602.5, -1123267.5 -1..."
4,198.750411,-1.809312e+06,7.091160e+05,200.159238,59.81225,198.750411,14427.999008,134873.872142,418.393202,Ice Shelf,...,0.698132,1.293517,0.509294,0.835543,0.500000,Lake,37,0.121964,9.98336,"POLYGON ((-1809304.184 709123.549, -1809311.33..."


## Write GeoParquet And Read It Back

Schema version 1.1.0 is preferred when the installed GeoPandas/pyarrow stack supports it. The fallback keeps the notebook runnable on older environments.


In [3]:
# To get effective geo parquet you have to use atleast schema=1.1.0 + sort the data spatially
# Infact if you have more  than 2gbs worth of data its a good idea to partition it spatially, not just sort it
# more info on best practices here: - https://geoparquet.io/
gdf.sort_values('geometry',  inplace=True, ignore_index=True)

gdf.to_parquet(
    PARQUET_FILE,
    engine='pyarrow',
    compression='zstd',
    write_covering_bbox=True, 
    schema_version='1.1.0'
)